# Sinh dữ liệu damage tốc độ cao bằng Gemini

Notebook tạo hai nhánh `zero_shot` và `few_shot` cho thí nghiệm Meta-Judge.
Prompt được Việt hoá từ Bảng 13 và 15 của paper; Level 1 dùng nhiễu dấu tiếng
Việt theo kế hoạch đồ án. Mỗi API key thuộc một project được chạy bởi một
worker riêng. Output JSONL giữ nguyên schema dùng cho notebook chấm metric và
có checkpoint để tiếp tục sau khi runtime bị ngắt.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()

# Hỗ trợ source.csv của đồ án hoặc references*.jsonl có id/source_zh/reference_vi.
INPUT_FILE = ROOT / "source.csv"
RUN_NAME = "paper-damage-vi-gemini-3-5-flash-lite-01"

GEMINI_MODEL = "gemini-3.5-flash-lite"
GEMINI_API_KEYS = [
    # Chỉ thêm key trong bản local; không commit API key vào Git.
]

PROMPT_TYPES = ("zero_shot", "few_shot")

# Khoảng câu 1-based, lấy cả hai đầu. None ở cuối nghĩa là chạy đến hết file.
SENTENCE_START = 1
SENTENCE_END = None

# Nhịp 12 RPM/project, chừa biên dưới mức 15 RPM thường gặp ở free tier.
REQUEST_INTERVAL_SECONDS = 5.0
REQUESTS_PER_DAY_PER_PROJECT = 500
MAX_OUTPUT_TOKENS = 400
# Không retry ở tầng HTTP. Một task lỗi tạm thời chỉ được đưa lại hàng đợi một lần.
MAX_TASK_ATTEMPTS = 2
# RPM/TPM được cooldown rồi thử lại một lần; RPD/billing dừng slot ngay.
MAX_RATE_LIMIT_RETRIES = 1
RATE_LIMIT_DEFAULT_WAIT_SECONDS = 60.0
RATE_LIMIT_MAX_WAIT_SECONDS = 65.0
REQUEST_TIMEOUT_SECONDS = 180
PROGRESS_EVERY = 10
CHECK_KEYS_BEFORE_RUN = True
# Request đầu tiên là một task thật; chỉ mở 19 worker sau khi model sinh thành công.
VALIDATE_MODEL_BEFORE_PARALLEL = True
DOWNLOAD_ARCHIVE = False

## 1. Bộ sinh damage và checkpoint

In [ ]:
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
import csv
import hashlib
import json
import math
import re
import shutil
import threading
import time
import urllib.error
import urllib.parse
import urllib.request


PROTOCOL = "paper_wmt_vi_one_level_per_request_v1"
DAMAGE_LEVELS = tuple(range(6))

WMT_BASE_PROMPT = """Bạn là một Bộ máy Làm hỏng Bản dịch.
Mục tiêu của bạn là nhận một 'bản_dịch_tham_chiếu' hoàn hảo và làm suy giảm nó đúng theo 'mức_hư_hại' được yêu cầu.

### QUY TẮC VỀ THÔNG TIN GỐC
1. **Nguồn sự thật:** 'câu_nguồn' và 'bản_dịch_tham_chiếu' xác định ý nghĩa chính xác.
2. **Tuân thủ nghiêm ngặt:** Không được cải thiện văn bản. Bạn phải chủ động làm hỏng nó đúng mức được yêu cầu.

### ĐẶC TẢ MỨC HƯ HẠI
Mức 0 (Diễn đạt lại): Viết lại 'bản_dịch_tham_chiếu' bằng từ đồng nghĩa hoặc cấu trúc câu khác. Kết quả PHẢI vẫn là một bản dịch chính xác, chất lượng cao của 'câu_nguồn' và hoàn toàn đúng ngữ pháp.
Mức 1 (Nhiễu bề mặt): Giữ nguyên nghĩa hoàn toàn, nhưng CHÈN LỖI HÌNH THỨC dễ thấy. Với tiếng Việt, dùng một trong các lỗi sau: bỏ dấu thanh ở 1–2 từ (ví dụ: "mười" → "muoi"), sai dấu thanh (ví dụ: "mà" → "má"), sai viết hoa hoặc thiếu dấu câu. Nghĩa phải giữ nguyên tuyệt đối, chỉ độ trôi chảy bị hỏng.
Mức 2 (Lược bỏ/Dịch thiếu): Loại bỏ một chi tiết hoặc sắc thái cụ thể có trong 'câu_nguồn', chẳng hạn bỏ một tính từ hoặc trạng từ. Bản dịch vẫn hiểu được nhưng phải thiếu thông tin rõ ràng so với bản tham chiếu.
Mức 3 (Lỗi ngữ nghĩa ở cấp độ từ): Dịch sai một từ mang nội dung như danh từ hoặc động từ thành một lựa chọn hợp lý nhưng không chính xác, chẳng hạn "ô tô" → "xe tải" hoặc "đi bộ" → "chạy". Đây phải là một lỗi cục bộ, cụ thể.
Mức 4 (Lỗi ngữ nghĩa nghiêm trọng): Thay đổi đáng kể ý nghĩa của toàn câu. Có thể đảo chủ thể và đối tượng, phủ định động từ chính hoặc thay đổi mạnh thời gian nếu việc đó làm câu mâu thuẫn với nguồn.
Mức 5 (Ảo giác/Thất bại hoàn toàn): Tạo một câu trôi chảy bằng tiếng Việt nhưng HOÀN TOÀN KHÔNG LIÊN QUAN đến 'câu_nguồn', hoặc là bản dịch của một đầu vào hoàn toàn khác.
"""

WMT_CONSTRAINTS = """### RÀNG BUỘC
1. NGÔN NGỮ ĐẦU RA: Kết quả phải cùng ngôn ngữ với 'bản_dịch_tham_chiếu', tức là tiếng Việt.
2. KHÔNG ĐƯỢC TRÙNG KHỚP HOÀN TOÀN: Từ Mức 1 trở lên, kết quả **KHÔNG ĐƯỢC** giống hệt 'bản_dịch_tham_chiếu'.
3. ĐỊNH DẠNG: Chỉ xuất đúng một chuỗi là bản dịch kết quả. Không thêm nhãn, giải thích, dấu ngoặc kép hoặc khối mã."""

WMT_EXAMPLES = """### VÍ DỤ

Người dùng:
câu_nguồn: 猫坐在垫子上。
bản_dịch_tham_chiếu: Con mèo ngồi trên tấm thảm.
mức_hư_hại: 0

Trợ lý:
Trên tấm thảm, con mèo đang ngồi.

Người dùng:
câu_nguồn: 她昨天买了一辆红色汽车。
bản_dịch_tham_chiếu: Hôm qua cô ấy đã mua một chiếc ô tô màu đỏ.
mức_hư_hại: 3

Trợ lý:
Hôm qua cô ấy đã mua một chiếc ô tô màu xanh.

Người dùng:
câu_nguồn: 技术正在快速发展。
bản_dịch_tham_chiếu: Công nghệ đang phát triển nhanh chóng.
mức_hư_hại: 5

Trợ lý:
Tôi thích ăn táo vào bữa sáng.
"""

WMT_ZERO_SHOT_PROMPT = f"{WMT_BASE_PROMPT}\n{WMT_CONSTRAINTS}\n"
WMT_FEW_SHOT_PROMPT = f"{WMT_BASE_PROMPT}\n{WMT_EXAMPLES}\n{WMT_CONSTRAINTS}\n"


class ProjectQuotaError(RuntimeError):
    def __init__(self, message, *, quota_kind="daily_or_billing"):
        super().__init__(message)
        self.quota_kind = quota_kind


class ProjectRateLimitError(RuntimeError):
    def __init__(self, message, *, quota_kind="rate", retry_after=None):
        super().__init__(message)
        self.quota_kind = quota_kind
        self.retry_after = retry_after


class GeminiRequestError(RuntimeError):
    def __init__(self, message, *, retryable=False):
        super().__init__(message)
        self.retryable = retryable


class ModelUnavailableError(RuntimeError):
    pass


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def unique_keys(values):
    if not isinstance(values, (list, tuple)):
        raise ValueError("GEMINI_API_KEYS phải là một mảng các chuỗi.")
    if any(not isinstance(value, str) for value in values):
        raise ValueError("Mỗi phần tử GEMINI_API_KEYS phải là chuỗi.")
    return list(dict.fromkeys(value.strip() for value in values if value.strip()))


def atomic_write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pending = path.with_suffix(path.suffix + ".tmp")
    pending.write_text(text, encoding="utf-8")
    pending.replace(path)


def write_json(path, value):
    atomic_write_text(
        path,
        json.dumps(value, ensure_ascii=False, indent=2) + "\n",
    )


def write_jsonl(path, rows):
    text = "".join(
        json.dumps(row, ensure_ascii=False) + "\n"
        for row in rows
    )
    atomic_write_text(path, text)


def read_json(path, default=None):
    path = Path(path)
    if not path.is_file():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def first_present(row, names):
    for name in names:
        value = row.get(name)
        if value is not None and str(value).strip():
            return str(value).strip()
    return ""


def normalize_reference(row, position):
    stt = first_present(row, ["STT", "stt"]) or str(position)
    identifier = first_present(row, ["id", "ID_cau_VLSP"])
    source = first_present(row, ["source_zh", "Nguon_ZH", "Cau_nguon_ZH"])
    reference = first_present(
        row,
        ["reference_vi", "Tham_chieu_VI", "Cau_tham_chieu_VI"],
    )
    if not identifier:
        try:
            identifier = f"vi-zh-2022-test-{int(float(stt)):04d}"
        except ValueError:
            identifier = f"row-{position:04d}"
    if not source or not reference:
        raise ValueError(
            f"Dòng {position} thiếu câu nguồn hoặc bản tham chiếu."
        )
    return {
        "id": identifier,
        "source_zh": source,
        "reference_vi": reference,
        "stt": stt,
    }


def load_references(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy input: {path}")
    if path.suffix.lower() == ".csv":
        with path.open(encoding="utf-8-sig", newline="") as stream:
            raw = list(csv.DictReader(stream))
    elif path.suffix.lower() in {".jsonl", ".ndjson"}:
        raw = [
            json.loads(line)
            for line in path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
    else:
        raise ValueError("INPUT_FILE phải là CSV hoặc JSONL.")
    rows = [normalize_reference(row, index) for index, row in enumerate(raw, 1)]
    ids = [row["id"] for row in rows]
    if len(ids) != len(set(ids)):
        raise ValueError("Input có id bị trùng.")
    return rows


def select_references(rows, start, end):
    if not isinstance(start, int) or start < 1:
        raise ValueError("SENTENCE_START phải là số nguyên dương.")
    stop = len(rows) if end is None else end
    if not isinstance(stop, int) or stop < start or stop > len(rows):
        raise ValueError(
            f"SENTENCE_END phải nằm trong khoảng {start}..{len(rows)}."
        )
    return rows[start - 1 : stop]


def system_prompt(prompt_type):
    prompts = {
        "zero_shot": WMT_ZERO_SHOT_PROMPT,
        "few_shot": WMT_FEW_SHOT_PROMPT,
    }
    try:
        return prompts[prompt_type]
    except KeyError:
        raise ValueError(
            "PROMPT_TYPES chỉ nhận zero_shot hoặc few_shot."
        ) from None


def render_messages(reference, prompt_type, damage_level):
    if prompt_type not in {"zero_shot", "few_shot"}:
        raise ValueError("PROMPT_TYPES chỉ nhận zero_shot hoặc few_shot.")
    if damage_level not in DAMAGE_LEVELS:
        raise ValueError("damage_level phải thuộc 0..5.")
    return [
        {"role": "system", "content": system_prompt(prompt_type)},
        {
            "role": "user",
            "content": (
                f"câu_nguồn: {reference['source_zh']}\n"
                f"bản_dịch_tham_chiếu: {reference['reference_vi']}\n"
                f"mức_hư_hại: {damage_level}"
            ),
        },
    ]


def request_payload(reference, prompt_type, damage_level, max_output_tokens):
    messages = render_messages(reference, prompt_type, damage_level)
    system = "\n\n".join(
        item["content"] for item in messages if item["role"] == "system"
    )
    user = "\n\n".join(
        item["content"] for item in messages if item["role"] != "system"
    )
    return {
        "systemInstruction": {"parts": [{"text": system}]},
        "contents": [{"role": "user", "parts": [{"text": user}]}],
        "generationConfig": {
            "temperature": 0.0,
            "maxOutputTokens": max_output_tokens,
        },
    }


def api_error_summary(detail):
    try:
        payload = json.loads(detail)
        error = payload.get("error", {})
        status = str(error.get("status", "")).strip()
        message = str(error.get("message", "")).strip()
        summary = ": ".join(part for part in (status, message) if part)
    except (AttributeError, TypeError, ValueError):
        summary = str(detail).strip()
    return re.sub(r"\s+", " ", summary)[:800] or "Không có chi tiết lỗi."


def seconds_from_duration(value):
    if isinstance(value, (int, float)):
        return max(0.0, float(value))
    match = re.fullmatch(r"\s*([0-9]+(?:\.[0-9]+)?)s\s*", str(value or ""))
    return float(match.group(1)) if match else None


def classify_429(detail):
    """Classify Gemini quota metadata without guessing from boilerplate text."""
    try:
        payload = json.loads(detail)
        error = payload.get("error", {})
    except (AttributeError, TypeError, ValueError):
        error = {}

    message = str(error.get("message", detail or ""))
    status = str(error.get("status", ""))
    code = str(error.get("code", ""))
    quota_fields = []
    retry_after = None

    details = error.get("details", [])
    if not isinstance(details, list):
        details = []
    for item in details:
        if not isinstance(item, dict):
            continue
        item_type = str(item.get("@type", "")).lower()
        if "retryinfo" in item_type:
            retry_after = seconds_from_duration(item.get("retryDelay"))
        violations = item.get("violations", [])
        if not isinstance(violations, list):
            continue
        for violation in violations:
            if not isinstance(violation, dict):
                continue
            quota_fields.extend(
                str(violation.get(name, ""))
                for name in ("quotaId", "quotaMetric")
            )

    if retry_after is None:
        retry_match = re.search(
            r"retry\s+in\s+([0-9]+(?:\.[0-9]+)?)s",
            message,
            flags=re.I,
        )
        if retry_match:
            retry_after = float(retry_match.group(1))

    quota_text = " ".join(quota_fields).lower()
    message_text = message.lower()
    code_text = f"{status} {code}".lower()
    daily_markers = (
        "perday",
        "per_day",
        "requestsperday",
        "tokensperday",
        "daily quota",
        "requests per day",
        "rpd",
    )
    rate_markers = (
        "perminute",
        "per_minute",
        "persecond",
        "per_second",
        "requestsperminute",
        "tokensperminute",
        "requests per minute",
        "rpm",
        "tpm",
    )
    billing_markers = (
        "prepayment credits are depleted",
        "prepay credit balance",
        "no credits",
        "billing account is not active",
    )

    if "quota_exceeded" in code_text or any(
        marker in quota_text or marker in message_text
        for marker in daily_markers
    ):
        kind = "rpd"
    elif any(marker in message_text for marker in billing_markers):
        kind = "billing"
    elif (
        "rate_limit_exceeded" in code_text
        or "too_many_requests" in code_text
        or any(marker in quota_text for marker in rate_markers)
    ):
        kind = "rpm_tpm"
    elif retry_after is not None:
        kind = "rpm_tpm"
    else:
        kind = "unknown"

    return {
        "kind": kind,
        "retry_after": retry_after,
        "summary": api_error_summary(detail),
    }


def post_gemini(key, model, payload, timeout):
    url = (
        "https://generativelanguage.googleapis.com/v1beta/models/"
        f"{urllib.parse.quote(model, safe='')}:generateContent"
    )
    body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    request = urllib.request.Request(
        url,
        data=body,
        headers={
            "Content-Type": "application/json",
            "x-goog-api-key": key,
        },
        method="POST",
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        if exc.code == 429:
            quota = classify_429(detail)
            message = f"Gemini HTTP 429 [{quota['kind']}]: {quota['summary']}"
            if quota["kind"] in {"rpd", "billing"}:
                raise ProjectQuotaError(
                    message,
                    quota_kind=quota["kind"],
                ) from None
            raise ProjectRateLimitError(
                message,
                quota_kind=quota["kind"],
                retry_after=quota["retry_after"],
            ) from None
        summary = api_error_summary(detail)
        if exc.code == 404:
            raise ModelUnavailableError(
                f"Gemini HTTP 404: {summary}"
            ) from None
        raise GeminiRequestError(
            f"Gemini HTTP {exc.code}: {summary}",
            retryable=exc.code in {408, 500, 502, 503, 504},
        ) from None
    except (urllib.error.URLError, TimeoutError) as exc:
        raise GeminiRequestError(
            f"Gemini network error: {exc}",
            retryable=True,
        ) from None


def extract_response_text(data):
    try:
        candidate = data["candidates"][0]
        finish_reason = candidate.get("finishReason", "STOP")
        parts = candidate["content"]["parts"]
        text = "".join(part.get("text", "") for part in parts).strip()
    except (KeyError, IndexError, TypeError) as exc:
        raise GeminiRequestError(
            f"Gemini response không hợp lệ: {data}",
            retryable=True,
        ) from exc
    if finish_reason not in {"STOP", "MAX_TOKENS"}:
        raise GeminiRequestError(f"Gemini dừng với finishReason={finish_reason}.")
    if finish_reason == "MAX_TOKENS":
        raise GeminiRequestError("Output bị cắt do MAX_OUTPUT_TOKENS quá thấp.")
    return text


def parse_prediction(text, reference_translation, damage_level):
    prediction = text.strip()
    if prediction.startswith("```"):
        prediction = re.sub(
            r"^```(?:text)?\s*|\s*```$",
            "",
            prediction,
            flags=re.I,
        ).strip()
    if (
        len(prediction) >= 2
        and prediction[0] == prediction[-1]
        and prediction[0] in {'"', "'"}
    ):
        prediction = prediction[1:-1].strip()
    if not prediction:
        raise GeminiRequestError("Gemini trả về output rỗng.", retryable=True)
    forbidden_labels = (
        "damage_level:",
        "mức_hư_hại:",
        "bản_dịch_tham_chiếu:",
    )
    if any(label in prediction.lower() for label in forbidden_labels):
        raise GeminiRequestError(
            "Output chứa nhãn thay vì chỉ bản dịch.",
            retryable=True,
        )
    if damage_level >= 1 and prediction == reference_translation.strip():
        raise GeminiRequestError(
            "Damage level từ 1 trở lên không được trùng bản tham chiếu.",
            retryable=True,
        )
    return prediction


def call_paper_damage(
    reference,
    prompt_type,
    damage_level,
    key,
    model,
    max_output_tokens,
    timeout,
):
    payload = request_payload(
        reference,
        prompt_type,
        damage_level,
        max_output_tokens,
    )
    response = post_gemini(key, model, payload, timeout)
    prediction = parse_prediction(
        extract_response_text(response),
        reference["reference_vi"],
        damage_level,
    )
    return prediction, response.get("usageMetadata", {})


def task_key(reference, prompt_type, damage_level):
    return f"{reference['id']}|{prompt_type}|{damage_level}"


def checkpoint_path(checkpoint_root, reference, prompt_type, damage_level):
    digest = hashlib.sha256(
        task_key(reference, prompt_type, damage_level).encode()
    ).hexdigest()[:24]
    return Path(checkpoint_root) / prompt_type / f"{digest}.json"


def build_row(reference, prompt_type, damage_level, model, prediction):
    return {
        "run_key": task_key(reference, prompt_type, damage_level),
        "id": reference["id"],
        "source_zh": reference["source_zh"],
        "reference_vi": reference["reference_vi"],
        "prediction_vi": prediction,
        "damage_level": damage_level,
        "prompt_type": prompt_type,
        "model_name": model,
        "backend": "gemini",
        "temperature": 0.0,
        "status": "ok",
        "error": None,
        "generation_protocol": PROTOCOL,
    }


def checkpoint_row(path, reference, prompt_type, damage_level, model):
    payload = read_json(path)
    if not isinstance(payload, dict):
        return None
    if (
        payload.get("protocol") != PROTOCOL
        or payload.get("model") != model
        or payload.get("prompt_type") != prompt_type
        or payload.get("damage_level") != damage_level
        or payload.get("id") != reference["id"]
    ):
        return None
    row = payload.get("row")
    if not isinstance(row, dict):
        return None
    if (
        row.get("status") != "ok"
        or row.get("damage_level") != damage_level
        or row.get("source_zh") != reference["source_zh"]
        or row.get("reference_vi") != reference["reference_vi"]
        or not str(row.get("prediction_vi", "")).strip()
    ):
        return None
    return row


def save_checkpoint(
    path,
    reference,
    prompt_type,
    damage_level,
    model,
    prediction,
    usage,
    worker_slot,
):
    write_json(
        path,
        {
            "protocol": PROTOCOL,
            "created_at": utc_now(),
            "id": reference["id"],
            "prompt_type": prompt_type,
            "damage_level": damage_level,
            "model": model,
            "worker_slot": worker_slot,
            "usage_metadata": usage,
            "row": build_row(
                reference,
                prompt_type,
                damage_level,
                model,
                prediction,
            ),
        },
    )


def model_access_status(key, slot, model, timeout):
    url = (
        "https://generativelanguage.googleapis.com/v1beta/models/"
        f"{urllib.parse.quote(model, safe='')}"
    )
    request = urllib.request.Request(
        url,
        headers={"x-goog-api-key": key},
        method="GET",
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            data = json.loads(response.read().decode("utf-8"))
        return {
            "slot": slot,
            "status": "model_visible",
            "model": data.get("name", model),
            "key": key,
        }
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        if exc.code == 429:
            quota = classify_429(detail)
            return {
                "slot": slot,
                "status": "model_check_429",
                "model": model,
                "quota_kind": quota["kind"],
                "note": "Vẫn thử một request sinh để phân loại quota chính xác.",
                "key": key,
            }
        return {
            "slot": slot,
            "status": "unavailable",
            "error": f"HTTP {exc.code}: {detail[:300]}",
        }
    except urllib.error.URLError as exc:
        return {
            "slot": slot,
            "status": "unavailable",
            "error": f"Network error: {exc}",
        }


def check_project_keys(keys, model, timeout):
    if not keys:
        return []
    with ThreadPoolExecutor(max_workers=len(keys)) as pool:
        futures = [
            pool.submit(model_access_status, key, index, model, timeout)
            for index, key in enumerate(keys, 1)
        ]
        results = [future.result() for future in futures]
    return sorted(results, key=lambda item: item["slot"])


def pending_tasks(references, prompt_types, checkpoint_root, model):
    tasks = []
    for reference in references:
        for prompt_type in prompt_types:
            for damage_level in DAMAGE_LEVELS:
                path = checkpoint_path(
                    checkpoint_root,
                    reference,
                    prompt_type,
                    damage_level,
                )
                if checkpoint_row(
                    path,
                    reference,
                    prompt_type,
                    damage_level,
                    model,
                ) is None:
                    tasks.append((reference, prompt_type, damage_level))
    return tasks


def finalize_outputs(references, prompt_types, checkpoint_root, generation_root, model):
    summary = {}
    for prompt_type in prompt_types:
        rows = []
        complete_ids = 0
        missing_run_keys = []
        for reference in references:
            sentence_complete = True
            for damage_level in DAMAGE_LEVELS:
                path = checkpoint_path(
                    checkpoint_root,
                    reference,
                    prompt_type,
                    damage_level,
                )
                completed = checkpoint_row(
                    path,
                    reference,
                    prompt_type,
                    damage_level,
                    model,
                )
                if completed is None:
                    sentence_complete = False
                    missing_run_keys.append(
                        task_key(reference, prompt_type, damage_level)
                    )
                else:
                    rows.append(completed)
            complete_ids += int(sentence_complete)
        output_path = Path(generation_root) / f"{prompt_type}.jsonl"
        write_jsonl(output_path, rows)
        summary[prompt_type] = {
            "status": "ready" if not missing_run_keys else "partial",
            "output": str(output_path),
            "sentences_complete": complete_ids,
            "rows": len(rows),
            "rows_expected": len(references) * len(DAMAGE_LEVELS),
            "rows_missing": len(missing_run_keys),
            "missing_run_keys": missing_run_keys,
        }
    return summary


def run_parallel_generation(
    references,
    prompt_types,
    keys,
    checkpoint_root,
    generation_root,
    *,
    model,
    request_interval,
    max_output_tokens,
    timeout,
    max_task_attempts,
    progress_every,
    max_rate_limit_retries=1,
    rate_limit_default_wait_seconds=60.0,
    rate_limit_max_wait_seconds=65.0,
    validate_model_first=True,
    sleeper=time.sleep,
    caller=call_paper_damage,
):
    if max_rate_limit_retries < 0:
        raise ValueError("max_rate_limit_retries không được âm.")
    if rate_limit_default_wait_seconds < 0 or rate_limit_max_wait_seconds < 0:
        raise ValueError("Thời gian chờ rate limit không được âm.")
    tasks = pending_tasks(references, prompt_types, checkpoint_root, model)
    queue = deque(tasks)
    condition = threading.Condition()
    attempts = {}
    failures = []
    disabled_slots = []
    completed = 0
    api_calls = 0
    task_requeues = 0
    rate_limit_waits = 0
    consecutive_rate_limits = {}
    fatal_error = None
    first_request_at = {}
    worker_entries = list(enumerate(keys, 1))
    workers_started = 0
    in_flight = 0
    started_at = time.monotonic()

    if tasks and validate_model_first:
        reference, prompt_type, damage_level = tasks[0]
        verified = False
        print(
            "Kiểm tra model bằng một task thật trước khi mở pool...",
            flush=True,
        )
        for slot, key in list(worker_entries):
            api_calls += 1
            request_started = time.monotonic()
            first_request_at[slot] = request_started
            try:
                prediction, usage = caller(
                    reference,
                    prompt_type,
                    damage_level,
                    key,
                    model,
                    max_output_tokens,
                    timeout,
                )
                save_checkpoint(
                    checkpoint_path(
                        checkpoint_root,
                        reference,
                        prompt_type,
                        damage_level,
                    ),
                    reference,
                    prompt_type,
                    damage_level,
                    model,
                    prediction,
                    usage,
                    slot,
                )
                queue.popleft()
                completed = 1
                verified = True
                print(
                    f"Model sinh thành công bằng slot {slot}; bắt đầu pool song song.",
                    flush=True,
                )
                break
            except ModelUnavailableError as exc:
                fatal_error = str(exc)
                failures.append(
                    {
                        "task": task_key(reference, prompt_type, damage_level),
                        "attempts": 1,
                        "error": fatal_error,
                    }
                )
                print(
                    "DỪNG TOÀN BỘ: model không dùng được; không mở worker pool. "
                    f"API calls={api_calls} | {fatal_error}",
                    flush=True,
                )
                break
            except ProjectQuotaError as exc:
                disabled_slots.append(
                    {
                        "slot": slot,
                        "quota_kind": exc.quota_kind,
                        "reason": str(exc),
                    }
                )
                worker_entries = [
                    entry for entry in worker_entries if entry[0] != slot
                ]
                print(
                    f"Preflight dừng slot {slot}: 429/{exc.quota_kind} "
                    f"| API calls={api_calls}",
                    flush=True,
                )
            except (ProjectRateLimitError, GeminiRequestError) as exc:
                print(
                    f"Preflight slot {slot} chưa qua: {type(exc).__name__} "
                    f"| API calls={api_calls} | {exc}",
                    flush=True,
                )
            except Exception as exc:
                print(
                    f"Preflight slot {slot} lỗi không dự kiến "
                    f"| API calls={api_calls} | {type(exc).__name__}: {exc}",
                    flush=True,
                )
        if not verified and fatal_error is None:
            fatal_error = (
                "Không project nào sinh thành công task kiểm tra; "
                "không mở worker pool."
            )
            failures.append(
                {
                    "task": task_key(reference, prompt_type, damage_level),
                    "attempts": api_calls,
                    "error": fatal_error,
                }
            )
            print(f"DỪNG AN TOÀN: {fatal_error}", flush=True)

    def take_task():
        nonlocal in_flight
        with condition:
            while not queue:
                if in_flight == 0:
                    return None
                condition.wait()
            task = queue.popleft()
            in_flight += 1
            return task

    def finish_task(requeue=None):
        nonlocal in_flight
        with condition:
            if requeue is not None:
                queue.append(requeue)
            in_flight -= 1
            condition.notify_all()

    def worker(slot, key):
        nonlocal api_calls, completed, fatal_error, rate_limit_waits, task_requeues
        next_request_at = first_request_at.get(slot, 0.0) + request_interval
        while True:
            task = take_task()
            if task is None:
                return
            reference, prompt_type, damage_level = task
            wait_seconds = next_request_at - time.monotonic()
            if wait_seconds > 0:
                time.sleep(wait_seconds)
            request_started = time.monotonic()
            next_request_at = request_started + request_interval
            with condition:
                api_calls += 1
            try:
                prediction, usage = caller(
                    reference,
                    prompt_type,
                    damage_level,
                    key,
                    model,
                    max_output_tokens,
                    timeout,
                )
                path = checkpoint_path(
                    checkpoint_root,
                    reference,
                    prompt_type,
                    damage_level,
                )
                save_checkpoint(
                    path,
                    reference,
                    prompt_type,
                    damage_level,
                    model,
                    prediction,
                    usage,
                    slot,
                )
                with condition:
                    completed += 1
                    consecutive_rate_limits[slot] = 0
                    if completed % max(1, progress_every) == 0:
                        elapsed = max(time.monotonic() - started_at, 0.001)
                        rate = completed * 60 / elapsed
                        print(
                            f"Hoàn tất {completed}/{len(tasks)} request mới "
                            f"| API calls={api_calls} "
                            f"| retry={task_requeues} "
                            f"| rate-wait={rate_limit_waits} "
                            f"| {rate:.1f} kết quả/phút",
                            flush=True,
                        )
                finish_task()
            except ProjectQuotaError as exc:
                with condition:
                    disabled_slots.append(
                        {
                            "slot": slot,
                            "quota_kind": exc.quota_kind,
                            "reason": str(exc),
                        }
                    )
                    task_requeues += 1
                    active_slots = len(keys) - len(disabled_slots)
                    call_count = api_calls
                print(
                    f"Dừng slot {slot}: HTTP 429/{exc.quota_kind} "
                    f"| còn {active_slots} slot "
                    f"| API calls={call_count}",
                    flush=True,
                )
                finish_task(requeue=task)
                return
            except ProjectRateLimitError as exc:
                with condition:
                    hit_count = consecutive_rate_limits.get(slot, 0) + 1
                    consecutive_rate_limits[slot] = hit_count
                    should_retry = hit_count <= max_rate_limit_retries
                    task_requeues += 1
                    if should_retry:
                        rate_limit_waits += 1
                    else:
                        disabled_slots.append(
                            {
                                "slot": slot,
                                "quota_kind": exc.quota_kind,
                                "reason": str(exc),
                            }
                        )
                    active_slots = len(keys) - len(disabled_slots)
                    call_count = api_calls
                finish_task(requeue=task)
                if not should_retry:
                    print(
                        f"Dừng slot {slot}: 429/{exc.quota_kind} lặp lại "
                        f"| còn {active_slots} slot "
                        f"| API calls={call_count}",
                        flush=True,
                    )
                    return

                suggested_wait = (
                    rate_limit_default_wait_seconds
                    if exc.retry_after is None
                    else exc.retry_after
                )
                wait_seconds = min(
                    max(suggested_wait, request_interval),
                    rate_limit_max_wait_seconds,
                )
                print(
                    f"Cooldown slot {slot}: 429/{exc.quota_kind} "
                    f"| chờ {wait_seconds:.1f}s rồi thử lại một lần "
                    f"| API calls={call_count}",
                    flush=True,
                )
                sleeper(wait_seconds)
                next_request_at = time.monotonic()
            except ModelUnavailableError as exc:
                with condition:
                    fatal_error = str(exc)
                    failures.append(
                        {
                            "task": task_key(reference, prompt_type, damage_level),
                            "attempts": 1,
                            "error": fatal_error,
                        }
                    )
                    queue.clear()
                print(
                    "DỪNG TOÀN BỘ: model không dùng được "
                    f"| API calls={api_calls} | {fatal_error}",
                    flush=True,
                )
                finish_task()
                return
            except GeminiRequestError as exc:
                key_name = task_key(reference, prompt_type, damage_level)
                with condition:
                    attempts[key_name] = attempts.get(key_name, 0) + 1
                    attempt = attempts[key_name]
                    should_retry = exc.retryable and attempt < max_task_attempts
                    if should_retry:
                        task_requeues += 1
                    else:
                        failures.append(
                            {
                                "task": key_name,
                                "attempts": attempt,
                                "error": str(exc),
                            }
                        )
                    call_count = api_calls
                    retry_count = task_requeues
                if should_retry:
                    print(
                        f"Retry task {key_name} lần {attempt + 1}/"
                        f"{max_task_attempts} | API calls={call_count} "
                        f"| retry={retry_count} | {exc}",
                        flush=True,
                    )
                    finish_task(requeue=task)
                else:
                    print(
                        f"Bỏ task {key_name} sau {attempt} lần gọi "
                        f"| API calls={call_count} | {exc}",
                        flush=True,
                    )
                    finish_task()
            except Exception as exc:
                key_name = task_key(reference, prompt_type, damage_level)
                with condition:
                    failures.append(
                        {
                            "task": key_name,
                            "attempts": 1,
                            "error": f"{type(exc).__name__}: {exc}",
                        }
                    )
                    call_count = api_calls
                print(
                    f"Bỏ task {key_name}: lỗi không dự kiến "
                    f"| API calls={call_count} | {type(exc).__name__}: {exc}",
                    flush=True,
                )
                finish_task()

    if tasks and fatal_error is None and worker_entries:
        workers_started = len(worker_entries)
        with ThreadPoolExecutor(max_workers=workers_started) as pool:
            futures = [
                pool.submit(worker, slot, key)
                for slot, key in worker_entries
            ]
            for future in futures:
                future.result()

    outputs = finalize_outputs(
        references,
        prompt_types,
        checkpoint_root,
        generation_root,
        model,
    )
    remaining = pending_tasks(references, prompt_types, checkpoint_root, model)
    return {
        "status": "ready" if not remaining else "partial",
        "protocol": PROTOCOL,
        "workers": len(keys),
        "workers_started_this_run": workers_started,
        "requests_pending_before_run": len(tasks),
        "requests_completed_this_run": completed,
        "api_calls_this_run": api_calls,
        "task_requeues_this_run": task_requeues,
        "rate_limit_waits_this_run": rate_limit_waits,
        "fatal_error": fatal_error,
        "requests_remaining": len(remaining),
        "disabled_project_slots": disabled_slots,
        "failed_tasks": failures,
        "branches": outputs,
    }

## 2. Kiểm tra input và hợp đồng chạy

In [ ]:
API_KEYS = unique_keys(GEMINI_API_KEYS)
if not API_KEYS:
    raise RuntimeError("Hãy thêm ít nhất một Gemini API key vào GEMINI_API_KEYS.")

all_references = load_references(INPUT_FILE)
references = select_references(
    all_references,
    SENTENCE_START,
    SENTENCE_END,
)

OUTPUT_ROOT = ROOT / "output" / RUN_NAME
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
GENERATION_ROOT = OUTPUT_ROOT / "generation"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

input_digest = hashlib.sha256(
    json.dumps(references, ensure_ascii=False, sort_keys=True).encode("utf-8")
).hexdigest()
contract = {
    "version": 2,
    "protocol": PROTOCOL,
    "input_file": str(INPUT_FILE.resolve()),
    "input_sha256": input_digest,
    "model": GEMINI_MODEL,
    "prompt_types": list(PROMPT_TYPES),
    "prompt_sha256": {
        prompt_type: hashlib.sha256(
            system_prompt(prompt_type).encode("utf-8")
        ).hexdigest()
        for prompt_type in PROMPT_TYPES
    },
    "sentence_start": SENTENCE_START,
    "sentence_end": SENTENCE_END,
    "max_output_tokens": MAX_OUTPUT_TOKENS,
    "temperature": 0.0,
}
contract_path = OUTPUT_ROOT / "experiment.json"
existing_contract = read_json(contract_path)
if existing_contract is not None and existing_contract != contract:
    raise RuntimeError(
        "Input/model/prompt đã đổi. Hãy đặt RUN_NAME mới để không trộn checkpoint."
    )
write_json(contract_path, contract)

print("Input:", INPUT_FILE)
print("Số câu được chọn:", len(references))
print("Prompt:", ", ".join(PROMPT_TYPES))
print("Số project key:", len(API_KEYS))
print("Output:", OUTPUT_ROOT)

## 3. Kiểm tra model trên từng project

Cell này chỉ kiểm tra key nhìn thấy model, không gọi sinh nội dung. Quota sinh
được xác nhận bằng request đầu tiên của mỗi worker ở bước 5.

In [ ]:
if CHECK_KEYS_BEFORE_RUN:
    key_status = check_project_keys(
        API_KEYS,
        GEMINI_MODEL,
        REQUEST_TIMEOUT_SECONDS,
    )
else:
    key_status = [
        {
            "slot": index,
            "status": "unchecked",
            "model": GEMINI_MODEL,
            "key": key,
        }
        for index, key in enumerate(API_KEYS, 1)
    ]

for item in key_status:
    printable = {key: value for key, value in item.items() if key != "key"}
    print(json.dumps(printable, ensure_ascii=False))

active_key_status = [
    item
    for item in key_status
    if item["status"] in {"model_visible", "model_check_429", "unchecked"}
]
ACTIVE_KEYS = [item["key"] for item in active_key_status]
if not ACTIVE_KEYS:
    raise RuntimeError("Không project nào truy cập được model đã chọn.")

## 4. Kế hoạch request

In [ ]:
tasks_before_run = pending_tasks(
    references,
    PROMPT_TYPES,
    CHECKPOINT_ROOT,
    GEMINI_MODEL,
)
effective_rpm = len(ACTIVE_KEYS) * 60 / REQUEST_INTERVAL_SECONDS
minimum_minutes = len(tasks_before_run) / effective_rpm if effective_rpm else math.inf
requests_per_project = math.ceil(len(tasks_before_run) / len(ACTIVE_KEYS))

print("Request còn lại:", len(tasks_before_run))
print("Giao thức:", PROTOCOL)
print("Thông lượng đặt trước:", f"{effective_rpm:.1f} request/phút")
print("Thời gian lý thuyết tối thiểu:", f"{minimum_minutes:.1f} phút")
print("Request/project ước tính:", requests_per_project)
if requests_per_project > REQUESTS_PER_DAY_PER_PROJECT:
    print(
        "CẢNH BÁO: số request/project vượt RPD cấu hình; "
        "thêm project key hoặc chạy tiếp sau khi quota reset."
    )

## 5. Sinh từng level theo paper trên các project song song

Chạy lại cell này với cùng `RUN_NAME` để tiếp tục các request chưa có checkpoint.
RPD/billing dừng slot ngay; RPM/TPM được cooldown rồi thử lại đúng một lần.

In [ ]:
generation_summary = run_parallel_generation(
    references,
    PROMPT_TYPES,
    ACTIVE_KEYS,
    CHECKPOINT_ROOT,
    GENERATION_ROOT,
    model=GEMINI_MODEL,
    request_interval=REQUEST_INTERVAL_SECONDS,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    timeout=REQUEST_TIMEOUT_SECONDS,
    max_task_attempts=MAX_TASK_ATTEMPTS,
    progress_every=PROGRESS_EVERY,
    max_rate_limit_retries=MAX_RATE_LIMIT_RETRIES,
    rate_limit_default_wait_seconds=RATE_LIMIT_DEFAULT_WAIT_SECONDS,
    rate_limit_max_wait_seconds=RATE_LIMIT_MAX_WAIT_SECONDS,
    validate_model_first=VALIDATE_MODEL_BEFORE_PARALLEL,
)
write_json(OUTPUT_ROOT / "generation_summary.json", generation_summary)
print(json.dumps(generation_summary, ensure_ascii=False, indent=2))

## 6. Kiểm tra và đóng gói output

In [ ]:
final_summary = finalize_outputs(
    references,
    PROMPT_TYPES,
    CHECKPOINT_ROOT,
    GENERATION_ROOT,
    GEMINI_MODEL,
)
not_ready = [
    prompt
    for prompt, result in final_summary.items()
    if result["status"] != "ready"
]
write_json(OUTPUT_ROOT / "final_audit.json", final_summary)

archive_base = ROOT / f"{RUN_NAME}-generation"
archive_path = Path(
    shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=OUTPUT_ROOT,
    )
)

print(json.dumps(final_summary, ensure_ascii=False, indent=2))
print("Archive:", archive_path)

if DOWNLOAD_ARCHIVE:
    try:
        from google.colab import files

        files.download(str(archive_path))
    except ImportError:
        print("Không phải Colab; tải archive từ đường dẫn đã in ở trên.")

if not_ready:
    raise RuntimeError(
        "Generation chưa hoàn tất: "
        + ", ".join(not_ready)
        + ". Chạy lại cell 5 sau khi quota sẵn sàng."
    )